## SARIMA
#### Seasonal AutoRegressive Integrated Moving Average
- seasonal time-series prediction model
- **NOT AVAILABLE** for multivariate time-series


In [8]:
# Import libraries
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from statsmodels.tsa.statespace.sarimax import SARIMAX
from pmdarima import auto_arima
from tqdm import tqdm

import warnings
from builtins import FutureWarning
warnings.filterwarnings("ignore", category=FutureWarning)


In [22]:
'''
Load dataset and preprocessing
-> train | test | submission | prediction
'''

# train data
train_data = pd.read_csv('./dataset/train/train.csv')
# ['date'] -> datetime
train_data['date'] = pd.to_datetime(train_data['date'], format='%Y-%m-%d')
# ordinal date feature
train_data['date_ordinal'] = train_data['date'].map(datetime.toordinal)
# store_menu_id
train_data['store_menu_id'] = train_data['store'] + "_" + train_data['menu']


# test data
for i in range(0, 10):
    test = pd.read_csv(f"./dataset/test/TEST_0{i}.csv")
    test['date'] = pd.to_datetime(test['date'], format='%Y-%m-%d')
    test['date_ordinal'] = test['date'].map(datetime.toordinal)
    test['store_menu_id'] = test['store'] + "_" + test['menu']
    # test_data_{i} for all test datasets
    globals()[f'test_data_{i}'] = test

# submission format
submission = pd.read_csv("./result/sample_submission_date.csv")

# Prediction result
all_preds = []


In [10]:
# Prediction function using SARIMA

def predict_with_auto_sarima(train_df, test_df, sid):
    # Select only the relevant store_menu_id
    train = train_df[train_df['store_menu_id'] == sid].copy()
    test = test_df[test_df['store_menu_id'] == sid].copy()

    # Combine
    data = pd.concat([train[['date_ordinal', 'sales']], test[['date_ordinal', 'sales']]], ignore_index=True)
    data = data.sort_values('date_ordinal').set_index('date_ordinal')

    # Define end of input
    input_end_ordinal = test['date_ordinal'].max()

    # Get last 28 days
    history = data.loc[:input_end_ordinal].tail(28)['sales']
    if len(history) != 28:
        raise ValueError(f"{sid} does not have exactly 28 days of data before prediction start.")


    # Train model (parameter tuning)
    model = auto_arima(
        history,
        start_p=0, start_q=0,
        max_p=2, max_q=2,
        seasonal=True,
        m=7,
        start_P=0, start_Q=0,
        max_P=2, max_Q=2,
        d=None, D=None,
        trace=False,
        error_action='ignore',
        suppress_warnings=True,
        stepwise=True,
        max_order=10
    )

    # Predict
    forecast = model.predict(n_periods=7)

    # Convert ordinal back to datetime
    forecast_ordinals = np.arange(input_end_ordinal + 1, input_end_ordinal + 8)
    forecast_dates = pd.to_datetime([datetime.fromordinal(int(o)) for o in forecast_ordinals])

    return pd.DataFrame({
        'date': forecast_dates,
        'store_menu_id': sid,
        'sales': forecast
    })


In [11]:
def run_recursive_forecasting(train_df, test_data_list):
    all_predictions = []

    for i, test_df in enumerate(test_data_list):
        test_df = test_df.copy()
        test_df['date'] = pd.to_datetime(test_df['date'])
        test_df = test_df.sort_values(['store_menu_id', 'date'])

        pred_list = []
        store_menu_ids = test_df['store_menu_id'].unique()

        for sid in tqdm(store_menu_ids, desc=f"Predicting TEST_{i}"):
            try:
                pred_df = predict_with_auto_sarima(train_df, test_df, sid)
                pred_list.append(pred_df)

                # Update train_df: add current test + prediction
                test_part = test_df[test_df['store_menu_id'] == sid]
                train_df = pd.concat([train_df, test_part, pred_df])

            except Exception as e:
                print(f"Failed for {sid}: {e}")

        if pred_list:
            all_predictions.append(pd.concat(pred_list))

    return pd.concat(all_predictions)


In [ ]:
# All test sets
test_data_list = [globals()[f'test_data_{i}'] for i in range(10)]

# Run full recursive prediction
final_preds = run_recursive_forecasting(train_data, test_data_list)

# time = 7m 45s


Predicting TEST_0:   0%|          | 0/193 [00:00<?, ?it/s]/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
Predicting TEST_0:   1%|          | 1/193 [00:00<02:45,  1.16it/s]/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
Predicting TEST_0:   1%|          | 2/193 [00:00<01:20,  2.37it/s]/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
Predicting TEST_0:   2%|▏         | 3/193 [00:02<02:26,  1.30it

Failed for 담하_(단체) 공깃밥: All lag values up to 'maxlag' produced singular matrices. Consider using a longer series, a different lag term or a different test.


/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
Predicting TEST_1:  14%|█▍        | 27/193 [00:03<00:19,  8.45it/s]/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
Predicting TEST_1:  15%|█▍        | 28/193 [00:03<00:19,  8.44it/s]/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supporte

Failed for 미라시아_보일링 랍스타 플래터(덜매운맛): All lag values up to 'maxlag' produced singular matrices. Consider using a longer series, a different lag term or a different test.


/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
Predicting TEST_2:  54%|█████▍    | 105/193 [00:22<00:13,  6.38it/s]/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
Predicting TEST_2:  55%|█████▍    | 106/193 [00:23<00:13,  6.55it/s]/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
Predicting TEST_2:  55%|█████▌    | 107/193 [00:23<00:21,  4.06it/s]/opt/anaconda3/envs/myenv/lib/python3.11/site-pac

...

/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
Predicting TEST_8:  51%|█████▏    | 99/193 [00:20<00:14,  6.57it/s]/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
Predicting TEST_8:  52%|█████▏    | 100/193 [00:21<00:17,  5.21it/s]/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
Predicting TEST_8:  52%|█████▏    | 101/193 [00:21<00:23,  3.89it/s]/opt/anaconda3/envs/myenv/lib/python3.11/site-pack

Failed for 미라시아_브런치 4인 패키지 : All lag values up to 'maxlag' produced singular matrices. Consider using a longer series, a different lag term or a different test.


/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
Predicting TEST_8:  55%|█████▍    | 106/193 [00:23<00:28,  3.04it/s]/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
Predicting TEST_8:  55%|█████▌    | 107/193 [00:24<00:34,  2.52it/s]/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
Predicting TEST_8:  56%|█████▌    | 108/193 [00:24<00:28,  2.97it/s]/opt/anaconda3/envs/myenv/lib/python3.11/site-pac

Failed for 느티나무 셀프BBQ_신라면: All lag values up to 'maxlag' produced singular matrices. Consider using a longer series, a different lag term or a different test.


/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
Predicting TEST_9:   6%|▌         | 11/193 [00:02<00:27,  6.60it/s]/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
Predicting TEST_9:   6%|▌         | 12/193 [00:02<00:27,  6.53it/s]/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supporte

In [15]:
final_preds

,date,store_menu_id,sales
28,2024-07-14,느티나무 셀프BBQ_1인 수저세트,1.973857
29,2024-07-15,느티나무 셀프BBQ_1인 수저세트,0.373271
30,2024-07-16,느티나무 셀프BBQ_1인 수저세트,1.516959
31,2024-07-17,느티나무 셀프BBQ_1인 수저세트,1.103401
32,2024-07-18,느티나무 셀프BBQ_1인 수저세트,2.322270
...,...,...,...
30,2025-05-27,화담숲카페_현미뻥스크림,33.073343
31,2025-05-28,화담숲카페_현미뻥스크림,32.535692
32,2025-05-29,화담숲카페_현미뻥스크림,18.334390
33,2025-05-30,화담숲카페_현미뻥스크림,25.029526


In [27]:
submission_updated = submission.copy()

for _, row in final_preds.iterrows():
    date = row['date']
    menu = row['store_menu_id']
    sales = max(0, row['sales'])  # Replace negative values with 0
    
    if date in submission_updated['date'].values and menu in submission_updated.columns:
        submission_updated.loc[submission_updated['date'] == date, menu] = sales

cols = ['date'] + [col for col in submission_updated.columns if col != 'date']
submission_updated = submission_updated[cols]
submission_updated = submission_updated.rename(columns={'date': '영업장명'})

submission_updated.to_csv('./cwj_SARIMA_tuned.csv', index=False, encoding='utf-8-sig')
